In [1]:
import os
import sys
import pandas as pd
import numpy as np

ruta_src = os.path.abspath(os.path.join("..", "src"))
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

from feature_engineering import (
    build_quant_features,
    quant_features,
)

from preprocessing import (
    temporal_split,
    fit_robust_params,
    transform_returns,
)
from representation_learning import build_autoencoder
input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

train_dev, val_dev = temporal_split(train)

robust_params = fit_robust_params(
    train_dev,
    return_cols
)

train_processed = transform_returns(
    train_dev,
    return_cols,
    robust_params
)

val_processed = transform_returns(
    val_dev,
    return_cols,
    robust_params
)

In [2]:
# Primero las 14 Quant Features:
# reconstruir quant +latent 
train_quant, val_quant, _, _ = build_quant_features(
    train_dev,
    val_dev,
    return_cols
)

X_train_quant = train_quant[quant_features].copy()
X_val_quant = val_quant[quant_features].copy()
from sklearn.impute import SimpleImputer

quant_imputer = SimpleImputer(strategy="median")

X_train_quant_imp = quant_imputer.fit_transform(X_train_quant)
X_val_quant_imp = quant_imputer.transform(X_val_quant)
#preparamos los retornos que recibe el autoencoder:
X_ae_train = train_processed[return_cols].to_numpy(dtype=np.float32)
X_ae_val = val_processed[return_cols].to_numpy(dtype=np.float32)
#construimos exactamente el AE de 05
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

autoencoder, encoder = build_autoencoder(
    input_dim=53,
    latent_dim=8
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)


MemoryError: Unable to allocate 272. MiB for an array with shape (673751, 53) and data type float64